# 10 - Monte Carlo & Stress Testing

**Author:** Sacha Huberty

**Purpose:** Build the forward-looking risk report for "the current
book" (the frozen Black-Litterman strategy's most recent target
weights): historical stress through the 2008 GFC, 2020 COVID crash,
and 2022 rate-hike selloff; a hypothetical per-asset-class sensitivity
table; a reverse stress test answering "what kills this strategy?";
and a 1,000-path GBM Monte Carlo fan chart with horizon histograms,
P(positive), and recovery-time analysis.

**Last updated:** 2026-07-26

**Scope note:** the 2008 scenario needs price history before the
project's `general.start_date` (2010-01-01), so this notebook
downloads a wider window (`risk.stress_data_start`, 2007-01-01) purely
for the stress/reverse-stress analysis. All 22 universe tickers have
real data by the acute crisis window used (2008-09-01 to 2009-03-31);
earlier 2008 dates are NOT used since some tickers (e.g. ACWI) don't
exist before 2008-03-28, and the study should not paper over that with
imputed history.

## Setup

In [ ]:
# Same fix as notebook 09: force single-threaded BLAS/OMP before
# numpy/scipy/tensorflow are imported, to avoid Windows thread-pool
# contention across hundreds of tiny per-week SLSQP/HMM calls.
import os

for _var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = "1"

import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, metrics, risk, simulation, strategy,
    universe,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["risk"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

# Wider window than the main backtest's start_date, needed for the 2008
# GFC scenario -- separate from the IS/OOS split used everywhere else.
prices = data.download_prices(tickers, start=cfg["risk"]["stress_data_start"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

is_end = pd.Timestamp(cfg["general"]["is_end_date"])
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]
returns.tail()

## Analysis / signal logic

### Recompute the frozen strategy: current book + OOS returns for the GBM

Reruns stage 9's frozen config (`meanreversion.lookback_days=20`,
`rebalance.no_trade_band=0.01`, `rebalance.max_weekly_turnover=0.01`)
through the same OOS backtest as notebook 09's part D, both as a
sanity cross-check (should reproduce very similar numbers) and to get
two things this notebook actually needs: (1) the most recent day's
actual (post-drift) weights, standing in for "the current book", and
(2) the OOS daily-return series used to estimate the GBM's drift/vol.

In [ ]:
buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

bt_cfg = copy.deepcopy(cfg)
bt_cfg["anomaly"]["epochs"] = 10
bt_cfg["anomaly"]["patience"] = 3
bt_cfg["anomaly"]["refit_frequency_days"] = 126

bl_fn = strategy.black_litterman_strategy(class_bucket, bt_cfg, posture_cfg)
bl_anomaly_fn = strategy.with_anomaly_override(bl_fn, bt_cfg)

frozen_result = backtest.run(bl_anomaly_fn, backtest_returns, bt_cfg)

current_book = frozen_result.weights.iloc[-1]
oos_returns = frozen_result.daily_returns.loc[oos_start:]

print("As-of date for current book:", frozen_result.weights.index[-1].date())
current_book.sort_values(ascending=False)

In [ ]:
frozen_metrics = {
    "ann_return": metrics.ann_return(oos_returns),
    "ann_vol": metrics.ann_vol(oos_returns),
    "sharpe": metrics.sharpe(oos_returns),
    "max_drawdown": metrics.max_drawdown(oos_returns),
}
print("Cross-check vs. notebook 09 (should be close):", frozen_metrics)

### Historical stress: the current book through 2008 / 2020 / 2022

Buy-and-hold the current book's weights through each scenario window,
no rebalancing: "if this shock repeated starting from today's actual
holdings, what happens?"

In [ ]:
historical = risk.historical_stress(current_book, returns, cfg["risk"]["scenarios"])
historical

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
historical["cumulative_return"].plot(kind="bar", ax=ax, color="firebrick")
ax.set_title("Historical stress: current book's cumulative return per scenario")
ax.set_ylabel("Cumulative return")
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Sensitivity stress: hypothetical per-asset-class shocks

Not a historical replay -- a direct "if equity dropped X% tomorrow,
right now, with today's weights" sensitivity table.

In [ ]:
sensitivity = risk.sensitivity_stress(
    current_book, class_bucket, cfg["risk"]["sensitivity_shocks"]
)
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = sensitivity["bucket"] + " " + (sensitivity["shock"] * 100).round(0).astype(int).astype(str) + "%"
ax.barh(labels, sensitivity["portfolio_impact"], color="darkorange")
ax.set_title("Sensitivity stress: portfolio impact of each hypothetical shock")
ax.set_xlabel("Portfolio impact")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

### Reverse stress: what kills this strategy?

Scans the current book's buy-and-hold return series across the full
2007-to-today history for its single worst day and worst rolling
`reverse_stress_window_days`-day window, and ranks which tickers drove
each by weight x realized return.

In [ ]:
reverse = risk.reverse_stress(
    current_book, returns, cfg["risk"]["reverse_stress_window_days"]
)
print("Worst single day:", reverse["worst_day"].date(), "return:", round(reverse["worst_day_return"], 4))
reverse["worst_day_contributors"]

In [ ]:
print(
    "Worst", cfg["risk"]["reverse_stress_window_days"], "trading-day window:",
    reverse["worst_window_start"].date(), "to", reverse["worst_window_end"].date(),
    "cumulative return:", round(reverse["worst_window_return"], 4),
)
reverse["worst_window_contributors"]

### Monte Carlo: 1,000-path GBM fan chart on the strategy's own OOS returns

Drift and vol are the frozen strategy's own realized OOS annualized
return/vol (not a per-asset model): "if the strategy keeps behaving
statistically like it has since 2022, what does the distribution of
outcomes look like going forward?"

In [ ]:
mu = frozen_metrics["ann_return"]
sigma = frozen_metrics["ann_vol"]
n_days = max(cfg["simulation"]["horizons_years"]) * simulation.TRADING_DAYS_PER_YEAR
sim_dates = simulation.business_days(returns.index[-1], n_days)

paths = simulation.gbm_paths(
    mu, sigma, sim_dates, cfg["simulation"]["n_scenarios"], cfg["general"]["random_seed"]
)
scenario_table = simulation.scenario_summary(paths, cfg["simulation"]["horizons_years"])
scenario_table

In [ ]:
all_dates = pd.DatetimeIndex([returns.index[-1]]).append(sim_dates)
percentiles = [5, 25, 50, 75, 95]
bands = np.percentile(paths, percentiles, axis=0)

fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(all_dates, bands[0], bands[4], color="steelblue", alpha=0.2, label="5-95th pct")
ax.fill_between(all_dates, bands[1], bands[3], color="steelblue", alpha=0.4, label="25-75th pct")
ax.plot(all_dates, bands[2], color="steelblue", linewidth=2, label="median")
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, label="breakeven")
ax.set_title(f"GBM fan chart ({cfg['simulation']['n_scenarios']} paths, mu={mu:.3f}, sigma={sigma:.3f})")
ax.set_ylabel("Growth of $1 from today")
ax.legend()
plt.tight_layout()
plt.show()

### Recovery-time analysis

`peak` is set to the growth factor needed to reclaim the frozen
strategy's OOS all-time high from today's level -- if the book is
currently in a drawdown, this asks "how long, and how likely, until
simulated paths get back to the prior peak?"

In [ ]:
equity = frozen_result.equity_curve.loc[oos_start:]
current_drawdown = float(equity.iloc[-1] / equity.cummax().iloc[-1] - 1.0)
peak_target = 1.0 / (1.0 + current_drawdown)
print(f"Current OOS drawdown from own peak: {current_drawdown:.2%}, recovery target growth factor: {peak_target:.4f}")

recovery = simulation.recovery_analysis(paths, peak=peak_target)
prob_recovered = recovery["recovered"].mean()
median_days = recovery.loc[recovery["recovered"], "recovery_days"].median()
print(f"P(recovers within {max(cfg['simulation']['horizons_years'])}y horizon): {prob_recovered:.2%}")
print(f"Median recovery time among paths that do recover: {median_days:.0f} trading days")

## Results

In [ ]:
print("=== Historical stress ===")
print(historical[["cumulative_return", "max_drawdown", "worst_day"]])
print()
print("=== Sensitivity stress (worst shock per bucket) ===")
print(sensitivity.loc[sensitivity.groupby("bucket")["portfolio_impact"].idxmin()])
print()
print("=== Reverse stress ===")
print("Worst day:", reverse["worst_day"].date(), round(reverse["worst_day_return"], 4))
print("Worst window:", reverse["worst_window_start"].date(), "-", reverse["worst_window_end"].date(), round(reverse["worst_window_return"], 4))
print()
print("=== Monte Carlo ===")
print(scenario_table)
print(f"P(recovers within horizon): {prob_recovered:.2%}, median recovery days: {median_days:.0f}")

## Notes / next steps

**Findings (as of the 2026-07-24 book):**

- **The cross-check against notebook 09 held up.** Recomputing the
  frozen strategy's OOS backtest here independently gave Sharpe 0.843
  (vs. 0.856 in notebook 09) and max drawdown -10.5% (vs. -10.4%) --
  close enough to be reassuring (the small gap is expected: a couple
  more trading days of live data plus normal autoencoder-refit
  nondeterminism), not a sign of a bug.
- **Today's actual book is fairly defensive:** the top holdings are
  BIL (14.7%), AGG (8.5%), SHY (6.0%), and BSV (5.8%) -- short-duration
  and investment-grade fixed income dominate, with the rest spread
  thinly across equity and commodity exposure (each roughly 3-5%).
  This reflects the utility gate currently favoring a defensive book
  over max_sharpe(mu_BL) more often than not.
- **Diversification reduces but does not eliminate crisis losses.**
  All three historical scenarios would hurt today's book meaningfully:
  GFC 2008 -17.6% cumulative (-22.6% max drawdown, the worst of the
  three), COVID 2020 -16.9% (-18.2%), and the 2022 rate-hike selloff
  -10.4% (-12.9%, the mildest, consistent with the current book's bond
  tilt actually helping less in a scenario driven by rate moves).
- **The reverse stress test independently rediscovers the COVID
  crash** as the single worst realized day (2020-03-16, -4.8%) and
  worst 20-day window (2020-02-20 to 2020-03-18, -18.2%) across the
  full 2007-2026 history available -- matching the hand-picked
  `covid_2020` scenario almost exactly. That's a good cross-
  validation that the scenario windows were sensibly chosen, not an
  independent new finding.
- **Monte Carlo, using the strategy's own realized OOS drift (4.4%)
  and vol (5.3%):** P(positive) rises from 80.3% at 1 year to 87.4%
  at 2 years and 95.3% at 4 years, as expected for a strategy with
  modest positive drift and low realized volatility.
- **Recovery analysis is a snapshot, not a fixed property of the
  strategy.** Today's book happens to sit only -2.5% below its own
  OOS peak, so the recovery read is undramatic: 98.4% probability of
  reclaiming that peak within 4 years, median 76 trading days among
  paths that do. Run this notebook again during an actual deep
  drawdown and this number will look very different -- that
  dependence on the current state is intentional (PROJECT_STRUCTURE's
  "weekly forward-looking risk report" framing), not a limitation of
  the analysis.
- **Scope note repeated for the record:** the 2008 scenario needed
  price history from 2007 (`risk.stress_data_start`), separate from
  the main backtest's 2010 `general.start_date`; a real caching bug in
  `data.py` (the cache key ignored the requested date range, silently
  returning whichever range had been cached first) was fixed as part
  of building this notebook, not left as a workaround.
